# Amharic ASR: reusable baseline workflow

This notebook orchestrates the `amharic_asr` package. Configure paths and hyperparameters in [configs/default.toml](../configs/default.toml). Use existing local audio and speaker-disjoint manifests; no dataset or research checkpoint is bundled.

The historical protocols and exact original notebook are under [research/](research/) and [archive/](archive/). This is a fresh FP32 baseline, not an automatic replay or exact resume of those experiments. Cells below have not been executed as part of the reorganization.


## 1. Locate the project and load configuration

Open from the project root or a subdirectory. No installation command is embedded in the notebook.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (
        candidate
        for candidate in candidates
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "amharic_asr").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Amharic-ASR project tree.")

source_dir = str(PROJECT_ROOT / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

from amharic_asr.config import load_config

config = load_config("configs/default.toml", project_root=PROJECT_ROOT)
config


## 2. Read local manifests and check speaker separation

See [the data contract](../docs/data-format.md) for required fields. Quarantine decisions and split selection must be made before this step.


In [ ]:
from amharic_asr.manifests import read_manifest, validate_speaker_separation

train_records = read_manifest(config.paths.train_manifest)
validation_records = read_manifest(config.paths.validation_manifest)
validate_speaker_separation(train_records, validation_records)
print(f"Training: {len(train_records)}; validation: {len(validation_records)}")


## 3. Create or reuse a processor

For a new baseline, vocabulary comes from training text only. For an existing model, point processor_dir to its original saved processor. Never rebuild a checkpoint vocabulary from a subset.


In [ ]:
from transformers import Wav2Vec2Processor

from amharic_asr.tokenization import create_processor

if config.paths.processor_dir.exists():
    processor = Wav2Vec2Processor.from_pretrained(str(config.paths.processor_dir))
else:
    processor = create_processor(
        train_records,
        config.paths.processor_dir,
        sample_rate=config.model.sample_rate,
    )
if processor.feature_extractor.sampling_rate != config.model.sample_rate:
    raise ValueError("Saved processor and configured sample rates differ.")

print(f"Vocabulary: {len(processor.tokenizer)}; blank ID: {processor.tokenizer.pad_token_id}")


## 4. Initialize XLS-R and local datasets

This step loads model weights and may download them if they are not cached. Set unfrozen_layers to 0 for head-only training or 2 for the final two encoder layers plus the head. The dataset checks actual decoded lengths for CTC feasibility.


In [ ]:
import torch
from torch.utils.data import DataLoader

from amharic_asr.data import AmharicASRDataset, CTCCollator
from amharic_asr.modeling import build_model
from amharic_asr.training import create_optimizer, resolve_device, seed_everything

seed_everything(config.training.seed)
device = resolve_device(config.training.device)
model = build_model(processor, config.model).to(device)

# For a compatible legacy partial checkpoint, explicitly restore its weights here.
# See docs/migration.md; use the same original baseline and saved processor.

dataset_options = {
    "audio_dir": config.paths.audio_dir,
    "processor": processor,
    "conv_kernel": model.config.conv_kernel,
    "conv_stride": model.config.conv_stride,
}
train_dataset = AmharicASRDataset(train_records, **dataset_options)
validation_dataset = AmharicASRDataset(validation_records, **dataset_options)
collator = CTCCollator(processor)
generator = torch.Generator().manual_seed(config.training.seed)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.training.batch_size,
    shuffle=True,
    generator=generator,
    collate_fn=collator,
    num_workers=0,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=config.training.batch_size,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
)
optimizer = create_optimizer(model, config.training)
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Device: {device}; trainable parameters: {trainable_count:,}")


## 5. Train and evaluate

Execute this section when the input data and configuration are ready. Each epoch exports a complete model and processor to a new folder. Use a new run_dir for a separate experiment. Exports do not include optimizer or RNG state for exact continuation. Stochastic layers are disabled by default to follow the later controlled experiments; this does not disable gradients.


In [ ]:
from amharic_asr.artifacts import export_model
from amharic_asr.evaluation import evaluate
from amharic_asr.training import train_epoch

config.paths.run_dir.mkdir(parents=True, exist_ok=False)
history = []
for epoch in range(1, config.training.epochs + 1):
    training_metrics = train_epoch(model, train_loader, optimizer, device, config.training)
    validation_metrics = evaluate(model, validation_loader, processor, device)
    history.append({
        "epoch": epoch,
        "training": training_metrics,
        "validation": validation_metrics,
    })
    destination = export_model(
        model,
        processor,
        config.paths.run_dir / f"epoch_{epoch:03d}",
        config,
        history,
    )
    print(
        f"Epoch {epoch}: CER={validation_metrics['cer']:.2%}, "
        f"WER={validation_metrics['wer']:.2%}; saved {destination}"
    )


## 6. Review results

CER and WER are stored as fractions. References and predictions follow validation-manifest order. Tune blank penalties or beam parameters only on development data; use a separate untouched test split for final results. Prefix beam search is available as amharic_asr.decoding.prefix_beam_decode for a single sequence with padded frames removed.


In [ ]:
latest = history[-1]["validation"]
examples = zip(
    validation_records, latest["references"], latest["predictions"], strict=True
)
for index, (record, reference, prediction) in enumerate(examples):
    if index == 5:
        break
    print(f"ID: {record['id']}")
    print(f"Reference: {reference}")
    print(f"Prediction: {prediction or '[EMPTY]'}")
    print()
